# 1. Exploratory Data Analysis (EDA)
## Digital Twin - Data Integration Project

**Objective**: Understand raw data before preprocessing  
**Input**: Raw datasets from `../Data/` folder  
**Output**: EDA findings to guide preprocessing decisions

---

### EDA Tasks:
1. Load all 4 raw datasets
2. Summary statistics
3. Missing value analysis
4. Distribution analysis
5. Outlier detection
6. Correlation analysis
7. Data quality assessment


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## 2. Set Data Paths

In [ ]:
# Paths relative to notebook location
DATA_DIR = Path('../Data')

# Dataset file names based on files present in Data/
DATA_FILES = {
    'heart_indicators': DATA_DIR / 'heart_disease_health_indicators_BRFSS2015.csv',
    'cardiovascular': DATA_DIR / 'cardio_train.csv',
    'diabetes_012': DATA_DIR / 'diabetes_012_health_indicators_BRFSS2015.csv',
    'diabetes_binary': DATA_DIR / 'diabetes_binary_health_indicators_BRFSS2015.csv',
    'diabetes_binary_5050split': DATA_DIR / 'diabetes_binary_5050split_health_indicators_BRFSS2015.csv'
}

# Check which files exist
print("Checking data files...")
for name, path in DATA_FILES.items():
    exists = "✓" if path.exists() else "✗"
    print(f"{exists} {name}: {path}")

## 3. Load Datasets

In [ ]:
datasets = {}

for name, path in DATA_FILES.items():
    try:
        if path.name == 'cardio_train.csv':
            df = pd.read_csv(path, sep=';')
        else:
            df = pd.read_csv(path)
        datasets[name] = df
        print(f"✓ Loaded {name}: {df.shape}")
    except FileNotFoundError:
        print(f"✗ File not found: {path}")
    except Exception as e:
        print(f"✗ Error loading {name}: {str(e)}")

print(f"\nTotal datasets loaded: {len(datasets)}")

## 4. Basic Dataset Information

In [ ]:
for name, df in datasets.items():
    print(f"\n{'='*70}")
    print(f"Dataset: {name.upper()}")
    print(f"{'='*70}")
    print(f"Shape: {df.shape} (rows × columns)")
    print(f"\nColumn Names:")
    print(df.columns.tolist())
    print(f"\nData Types:")
    print(df.dtypes)
    print(f"\nFirst 3 rows:")
    print(df.head(3))

## 4.5. Additional Visualizations

In [ ]:
# Visualization 1: Target/Class Distribution
n_datasets = len(datasets)
n_cols = 2
n_rows = int(np.ceil(n_datasets / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes = np.array(axes).ravel()

for ax, (name, df) in zip(axes, datasets.items()):
    target_col = df.columns[0]
    counts = df[target_col].value_counts().sort_index()
    bars = ax.bar(counts.index.astype(str), counts.values, color='#4C72B0')
    ax.set_title(f'{name} - Target Distribution ({target_col})')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')

    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f'{int(height):,}',
            ha='center',
            va='bottom',
            fontsize=9
        )

for ax in axes[len(datasets):]:
    ax.axis('off')

plt.tight_layout()
plt.show()
print('✓ Target distribution visualization complete')

In [ ]:
# Visualization 2: Correlation Heatmaps for Numeric Features
numeric_datasets = {}

for name, df in datasets.items():
    numeric_df = df.select_dtypes(include=['int64', 'float64']).copy()
    if 'id' in numeric_df.columns:
        numeric_df = numeric_df.drop(columns=['id'])
    if numeric_df.shape[1] > 1:
        numeric_datasets[name] = numeric_df

if numeric_datasets:
    n = len(numeric_datasets)
    n_cols = 2
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
    axes = np.array(axes).ravel()

    for ax, (name, numeric_df) in zip(axes, numeric_datasets.items()):
        corr = numeric_df.corr()
        sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0, cbar=False)
        ax.set_title(f'{name} - Correlation Heatmap')

    for ax in axes[len(numeric_datasets):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()
    print('✓ Correlation heatmap visualization complete')
else:
    print('No numeric datasets available for correlation heatmaps.')

## 5. Summary Statistics

In [ ]:
for name, df in datasets.items():
    print(f"\n{'='*70}")
    print(f"{name.upper()} - Summary Statistics")
    print(f"{'='*70}")
    print(df.describe())

## 6. Missing Value Analysis

In [ ]:
def analyze_missing(df, name):
    """Analyze missing values in dataset"""
    missing = pd.DataFrame({
        'Column': df.columns,
        'Missing_Count': df.isnull().sum(),
        'Missing_%': (df.isnull().sum() / len(df) * 100).round(2)
    })
    missing = missing[missing['Missing_Count'] > 0].sort_values('Missing_%', ascending=False)
    
    print(f"\n{name}:")
    if len(missing) == 0:
        print("  No missing values found!")
    else:
        print(missing.to_string(index=False))
    
    return missing

missing_analysis = {}
for name, df in datasets.items():
    missing_analysis[name] = analyze_missing(df, name)

## 7. Data Type Distribution

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}:")
    print(f"  Numeric columns: {df.select_dtypes(include=['float64', 'int64']).shape[1]}")
    print(f"  Categorical columns: {df.select_dtypes(include=['object']).shape[1]}")
    print(f"  Bool columns: {df.select_dtypes(include=['bool']).shape[1]}")

## 8. Numeric Columns Distribution

In [ ]:
# Analyze numeric columns
for name, df in datasets.items():
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    
    if numeric_cols:
        print(f"\n{name} - Numeric Columns Distribution:")
        print(f"Total numeric columns: {len(numeric_cols)}")
        
        # Show distribution for first few numeric columns
        for col in numeric_cols[:5]:
            print(f"\n  {col}:")
            print(f"    Min: {df[col].min()}, Max: {df[col].max()}")
            print(f"    Mean: {df[col].mean():.2f}, Median: {df[col].median():.2f}")
            print(f"    Std: {df[col].std():.2f}")

## 9. Categorical Columns Distribution

In [ ]:
# Analyze categorical columns
for name, df in datasets.items():
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    
    if cat_cols:
        print(f"\n{name} - Categorical Columns:")
        for col in cat_cols[:5]:
            print(f"\n  {col}:")
            print(f"    Unique values: {df[col].nunique()}")
            print(df[col].value_counts().head())

## 10. Outlier Detection (IQR Method)

In [ ]:
def detect_outliers(df, column):
    """Detect outliers using IQR method"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return ((df[column] < lower) | (df[column] > upper)).sum()

for name, df in datasets.items():
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    
    if numeric_cols:
        print(f"\n{name} - Outliers (IQR method):")
        outlier_counts = {}
        for col in numeric_cols:
            outlier_counts[col] = detect_outliers(df, col)
        
        # Show columns with outliers
        outliers_df = pd.DataFrame(list(outlier_counts.items()), columns=['Column', 'Outlier_Count'])
        outliers_df = outliers_df[outliers_df['Outlier_Count'] > 0].sort_values('Outlier_Count', ascending=False)
        
        if len(outliers_df) > 0:
            print(outliers_df.to_string(index=False))
        else:
            print("  No outliers detected!")

## 11. Correlation Analysis

In [ ]:
for name, df in datasets.items():
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    
    if len(numeric_cols) > 1:
        print(f"\n{name} - Top Correlations:")
        corr = df[numeric_cols].corr().abs()
        
        # Get pairs with high correlation
        corr_pairs = []
        for i in range(len(corr.columns)):
            for j in range(i+1, len(corr.columns)):
                if corr.iloc[i, j] > 0.7:  # High correlation threshold
                    corr_pairs.append({
                        'Column1': corr.columns[i],
                        'Column2': corr.columns[j],
                        'Correlation': corr.iloc[i, j]
                    })
        
        if corr_pairs:
            corr_df = pd.DataFrame(corr_pairs).sort_values('Correlation', ascending=False)
            print(corr_df.to_string(index=False))
        else:
            print("  No high correlations (>0.7) found")

## 12. Duplicate Records

In [ ]:
for name, df in datasets.items():
    dup_count = df.duplicated().sum()
    dup_pct = (dup_count / len(df) * 100) if len(df) > 0 else 0
    print(f"{name}: {dup_count} duplicates ({dup_pct:.2f}%)")

## 13. Data Quality Summary

In [ ]:
quality_summary = []

for name, df in datasets.items():
    quality_summary.append({
        'Dataset': name,
        'Records': len(df),
        'Columns': len(df.columns),
        'Missing_%': round((df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100), 2),
        'Duplicates': df.duplicated().sum(),
        'Memory_MB': round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    })

quality_df = pd.DataFrame(quality_summary)
print("\nDATA QUALITY SUMMARY:")
print(quality_df.to_string(index=False))

## 14. Visualizations

In [ ]:
# Visualization 1: Missing Values
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Data Quality Overview - Missing Values', fontsize=16, fontweight='bold')

for idx, (name, df) in enumerate(datasets.items()):
    if idx < 4:
        ax = axes[idx // 2, idx % 2]
        missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
        missing_pct = missing_pct[missing_pct > 0]
        
        if len(missing_pct) > 0:
            missing_pct.plot(kind='barh', ax=ax, color='coral')
            ax.set_title(f'{name} - Missing Values (%)')
            ax.set_xlabel('Missing %')
        else:
            ax.text(0.5, 0.5, 'No Missing Values', ha='center', va='center')
            ax.set_title(f'{name} - No Missing Values')

plt.tight_layout()
plt.show()
print("✓ Missing values visualization complete")

In [ ]:
# Visualization 2: Dataset Size Comparison
fig, ax = plt.subplots(figsize=(10, 6))

names = list(datasets.keys())
records = [len(df) for df in datasets.values()]

ax.bar(names, records, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
ax.set_ylabel('Number of Records')
ax.set_title('Dataset Size Comparison')
ax.set_xticklabels(names, rotation=45, ha='right')

# Add value labels on bars
for i, v in enumerate(records):
    ax.text(i, v + 5000, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.show()
print("✓ Dataset size visualization complete")

## 15. EDA Summary & Insights

In [ ]:
print("\n" + "="*70)
print("EDA SUMMARY & RECOMMENDATIONS FOR PREPROCESSING")
print("="*70)

print("\n✓ KEY FINDINGS:")
print(f"  1. Total Records: {sum(len(df) for df in datasets.values()):,}")
print(f"  2. Data Sources: {len(datasets)}")
print(f"  3. Total Columns: {sum(len(df.columns) for df in datasets.values())}")

print("\n✓ MISSING VALUES:")
for name, df in datasets.items():
    missing_pct = (df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100)
    print(f"  - {name}: {missing_pct:.2f}%")

print("\n✓ DATA QUALITY:")
for name, df in datasets.items():
    duplicates = df.duplicated().sum()
    print(f"  - {name}: {duplicates} duplicates")

print("\n✓ NEXT STEPS (PREPROCESSING):")
print("  1. Standardize schemas to unified format")
print("  2. Handle missing values (KNN imputation for numeric, mode for categorical)")
print("  3. Detect and treat outliers")
print("  4. Feature engineering (derive clinical features)")
print("  5. Normalize numeric features")
print("  6. Encode categorical variables")
print("\n" + "="*70)